# 01 - Build Audited KVS Method Data

[Open in Google Colab](https://colab.research.google.com/github/zhaoqyu/Lab-NLP/blob/mike/alignment_benchmark/notebooks/01_build_data.ipynb)

**Objective.** Create one auditable canonical pair per KVS source, then derive fair views for every method.

Use a GPU runtime. Persistent artifacts are written to Google Drive, so interrupted Colab sessions can resume. Run cells from top to bottom.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Select a GPU runtime first.'


In [ ]:
REPO_URL = 'https://github.com/zhaoqyu/Lab-NLP.git'
BRANCH = 'mike'
REPO_DIR = Path('/content/Lab-NLP')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', 'alignment_benchmark/requirements-colab.txt'],
    check=True,
)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', './alignment_benchmark', '--no-deps'],
    check=True,
)


In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/Lab-NLP/valuebench-paper')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['VALUEBENCH_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
CONFIG = 'alignment_benchmark/configs/paper.yaml'

def run(*arguments: str) -> None:
    command = ['valuebench', *arguments, '--config', CONFIG]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

print('Persistent output:', OUTPUT_ROOT)


In [ ]:
from google.colab import userdata

api_key = userdata.get('OPENROUTER_API_KEY')
assert api_key, 'Add OPENROUTER_API_KEY in the Colab Secrets panel.'
os.environ['OPENROUTER_API_KEY'] = api_key
del api_key
print('OpenRouter secret loaded without displaying it.')


In [ ]:
run('doctor', '--strict')


## 1. Preview the Teacher contract

This writes one pending prompt locally without making a paid request. Inspect it before the full run.

In [ ]:
run('teacher', '--dry-run', '--limit', '1')

jobs = OUTPUT_ROOT / 'data' / 'teacher_jobs.jsonl'
print(jobs.read_text(encoding='utf-8')[:4000])


## 2. Generate all canonical records

The run is atomic and resumable. It requests strict JSON and stores no private chain-of-thought. OpenRouter usage charges apply.

In [ ]:
run('teacher')


## 3. Audit fidelity and prepare the external test set

In [ ]:
run('audit-data', '--fail-on-quality')
run('prepare-aita')


## 4. Score the frozen base model

These ratings become SFT control labels. Preference margins provide the HyPO mismatch diagnostic.

In [ ]:
run('collect-baselines')
run('summarize-mismatch')


## 5. Build and verify all equal-source method views

In [ ]:
run('build-views')
run('validate-data')
run('make-plan')
run('status')


## Checkpoint

The canonical data, reports, baselines, method views, and experiment plan now live on Drive. Continue with notebook 02.